# E-GraphSAGE on Edge-IIoTset — all 2.2M flows

Implements **E-GraphSAGE** (Lo et al., *IEEE/IFIP NOMS 2022*, arXiv:2103.16329)
directly from the paper, in plain PyTorch. No `torch-geometric`, so the only
install is `pip install torch`.

---

### The graph

| | |
|---|---|
| **Node** | a host (an IP endpoint) |
| **Edge** | a flow between two hosts, carrying the 38 flow features |
| **Prediction** | one per flow — attack or benign |

Node features start as a constant vector of ones. The paper does this on purpose:
an IP address must never become a feature, or the model simply memorises the
attacker's address and learns nothing that transfers to another network.

### Two things make the result defensible

**An MLP control.** Same features, same labels, same split, no graph. Any gap
between it and E-GraphSAGE is the graph's doing and nothing else. Without this,
a higher number proves nothing — it could be the extra parameters alone.

**A zero-day split.** Whole attack families are held out of training. The
autoencoder never saw an attack at all, so a supervised model that had seen every
family would win unfairly.


## 0 — Configuration

`WINDOW` is the parameter that matters most, and it is worth understanding before
changing it.

Flows are cut into windows of consecutive rows, and each window becomes its own
small graph. In a **large** window a host sends attack and benign traffic alike,
so its embedding is an average of the two and says little. In a **small** window
the host's embedding reflects one moment.

| Window | Flows per host | MLP F1 | E-GraphSAGE F1 |
|---|---|---|---|
| **100** | 2.1 | 34.1% | **95.4%** |
| 250 | 5.0 | 30.9% | 87.6% |
| 500 | 10.0 | 26.7% | 67.4% |
| 2000 | 40.0 | 14.6% | 22.3% |

*(measured on synthetic flows built so the features alone carry no signal — section 8
reruns this sweep on the real data)*


In [ ]:
CSV_PATH = r'C:\Users\KimoStore\Desktop\PoC\DNN-EdgeIIoT-dataset.csv'

SAMPLE_ROWS = None     # None = every flow in the file
WINDOW      = 100      # flows per graph
BATCH_WIN   = 400      # windows per training batch -> memory stays flat

HELDOUT_ATTACKS = ['XSS', 'SQL_injection', 'Backdoor']   # never seen in training

EPOCHS, LR, HIDDEN, DROPOUT = 40, 0.005, 64, 0.2
SEED    = 42
OUT_DIR = 'EGraphSAGE_results'

import os, gc, time, json
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as Fn

np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(os.cpu_count())
os.makedirs(OUT_DIR, exist_ok=True)
print('torch', torch.__version__, '| threads', torch.get_num_threads())
assert os.path.exists(CSV_PATH), f'CSV not found: {CSV_PATH}'


## 1 — Load every flow

Read in chunks, numbers converted leniently so a column that turns to text
part-way through the file cannot halt the load. `ip.src_host` and `ip.dst_host`
are loaded this time — to wire the graph, never as features.


In [ ]:
peek = pd.read_csv(CSV_PATH, nrows=2000, low_memory=False)
LABEL_COL, TYPE_COL = 'Attack_label', 'Attack_type'
SRC_COL,  DST_COL   = 'ip.src_host', 'ip.dst_host'
for col in (LABEL_COL, TYPE_COL, SRC_COL, DST_COL):
    assert col in peek.columns, f'missing column: {col}'

ID_LIKE = {'frame.time','ip.src_host','ip.dst_host','arp.src.proto_ipv4',
           'arp.dst.proto_ipv4','http.file_data','http.request.full_uri',
           'http.request.version','icmp.transmit_timestamp','tcp.options',
           'tcp.payload','mqtt.msg','dns.qry.name','http.referer','mqtt.topic'}

numeric_cols = [col for col in peek.columns
                if col not in (LABEL_COL, TYPE_COL) and col not in ID_LIKE
                and pd.api.types.is_numeric_dtype(peek[col])]
keep = numeric_cols + [LABEL_COL, TYPE_COL, SRC_COL, DST_COL]
print(f'{len(numeric_cols)} candidate numeric features')
del peek; gc.collect()


In [ ]:
t0 = time.time()
parts, unparsable = [], {col: 0 for col in numeric_cols}

for chunk in pd.read_csv(CSV_PATH, usecols=keep, chunksize=200_000, low_memory=False):
    for col in numeric_cols:
        conv = pd.to_numeric(chunk[col], errors='coerce')
        unparsable[col] += int(conv.isna().sum() - chunk[col].isna().sum())
        chunk[col] = conv.astype(np.float32)
    chunk[LABEL_COL] = (pd.to_numeric(chunk[LABEL_COL], errors='coerce')
                          .fillna(0).astype(np.int8))
    parts.append(chunk)
    if SAMPLE_ROWS and sum(map(len, parts)) >= SAMPLE_ROWS:
        break

df = pd.concat(parts, ignore_index=True)
if SAMPLE_ROWS:
    df = df.iloc[:SAMPLE_ROWS]
del parts; gc.collect()

print(f'loaded {len(df):,} flows in {time.time()-t0:.0f}s')
print(f'memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')


In [ ]:
n = len(df)
dead = [col for col, v in unparsable.items() if v > 0.01 * n]
if dead:
    print(f'dropping {len(dead)} text columns: {dead}')
    numeric_cols = [col for col in numeric_cols if col not in dead]

print(f'features: {len(numeric_cols)}')
print()
print(df[TYPE_COL].value_counts())

missing = [a for a in HELDOUT_ATTACKS if a not in set(df[TYPE_COL].unique())]
assert not missing, f'held-out families not in the data: {missing}'


## 2 — Windowed graphs

Each window gets its own, disjoint set of host nodes. Windows therefore share no
nodes at all, which means a batch of windows is an **exact** subgraph — batching
here is not an approximation, nothing is lost.


In [ ]:
def build_windowed_graph(src_codes, dst_codes, window):
    """Returns per-flow endpoint node ids, the window each flow sits in,
    and the total node count."""
    n = len(src_codes)
    flow_src = np.empty(n, dtype=np.int64)
    flow_dst = np.empty(n, dtype=np.int64)
    win_id   = np.empty(n, dtype=np.int64)
    offset = 0

    for w, begin in enumerate(range(0, n, window)):
        end = min(begin + window, n)
        s, d = src_codes[begin:end], dst_codes[begin:end]
        _, inv = np.unique(np.concatenate([s, d]), return_inverse=True)
        k = len(s)
        flow_src[begin:end] = inv[:k] + offset
        flow_dst[begin:end] = inv[k:] + offset
        win_id[begin:end]   = w
        offset += int(inv.max()) + 1

    return flow_src, flow_dst, win_id, offset


In [ ]:
src_codes = pd.factorize(df[SRC_COL].astype(str))[0]
dst_codes = pd.factorize(df[DST_COL].astype(str))[0]
print(f'distinct hosts in the capture: sources {src_codes.max()+1}, destinations {dst_codes.max()+1}')

t0 = time.time()
flow_src, flow_dst, win_id, n_nodes = build_windowed_graph(src_codes, dst_codes, WINDOW)
n_win = int(win_id.max()) + 1
print(f'built in {time.time()-t0:.0f}s')
print(f'  windows {n_win:,}  |  nodes {n_nodes:,}  |  {n_nodes/n_win:.1f} hosts per window')
print(f'  flows per host inside a window: {WINDOW/(n_nodes/n_win):.1f}')
del src_codes, dst_codes; gc.collect()


## 3 — Features and the zero-day split

Windows are split, not individual flows: a flow and its context must stay on the
same side of the divide, otherwise the model sees test neighbours while training.

Windows containing a held-out family go to the zero-day set regardless.


In [ ]:
from sklearn.preprocessing import StandardScaler

X = df[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
const = [col for col in X.columns if X[col].nunique() <= 1]
X = X.drop(columns=const)
print(f'dropped {len(const)} constant columns -> {X.shape[1]} features')

y     = df[LABEL_COL].values.astype(np.float32)
atype = df[TYPE_COL].values.astype(str)
Xv    = X.values.astype(np.float32)
del X; gc.collect()


In [ ]:
rng = np.random.default_rng(SEED)

heldout_flow = np.isin(atype, HELDOUT_ATTACKS)
win_has_heldout = np.zeros(n_win, dtype=bool)
win_has_heldout[np.unique(win_id[heldout_flow])] = True

clean_wins = np.where(~win_has_heldout)[0]
rng.shuffle(clean_wins)
cut = int(0.7 * len(clean_wins))
train_wins = np.sort(clean_wins[:cut])
test_wins  = np.sort(clean_wins[cut:])
zero_wins  = np.where(win_has_heldout)[0]

in_train = np.isin(win_id, train_wins)
scaler = StandardScaler().fit(Xv[in_train])
Xv = scaler.transform(Xv).astype(np.float32)

print(f'train windows      {len(train_wins):,}  flows {int(in_train.sum()):,}')
print(f'test windows       {len(test_wins):,}')
print(f'zero-day windows   {len(zero_wins):,}')

assert not np.isin(atype[in_train], HELDOUT_ATTACKS).any(), \
    'LEAK: a held-out family reached training'
print('\nno held-out family in training — split is clean')


## 4 — The models

Written out in full, so every line is inspectable and the only dependency is
PyTorch.


In [ ]:
def mean_aggregate(messages, index, num_nodes):
    out = torch.zeros(num_nodes, messages.size(1), dtype=messages.dtype)
    out.index_add_(0, index, messages)
    deg = torch.zeros(num_nodes, dtype=messages.dtype)
    deg.index_add_(0, index, torch.ones(index.size(0), dtype=messages.dtype))
    return out / deg.clamp(min=1).unsqueeze(1)

class EGraphSAGELayer(nn.Module):
    """h_N(v) = MEAN(CONCAT(h_u, e_uv));  h_v = relu(W . CONCAT(h_v, h_N(v)))"""
    def __init__(self, node_dim, edge_dim, out_dim, dropout=0.2):
        super().__init__()
        self.lin = nn.Linear(node_dim * 2 + edge_dim, out_dim)
        self.dropout = dropout
    def forward(self, h, e, mp_src, mp_dst, mp_feat):
        msg = torch.cat([h[mp_src], e[mp_feat]], dim=1)
        agg = mean_aggregate(msg, mp_dst, h.size(0))
        return Fn.dropout(Fn.relu(self.lin(torch.cat([h, agg], dim=1))),
                          self.dropout, self.training)

class EGraphSAGE(nn.Module):
    def __init__(self, edge_dim, hidden=64, dropout=0.2, use_edge_in_head=True):
        super().__init__()
        self.embed0 = nn.Linear(1, hidden)      # constant ones -> node vector
        self.l1 = EGraphSAGELayer(hidden, edge_dim, hidden, dropout)
        self.l2 = EGraphSAGELayer(hidden, edge_dim, hidden, dropout)
        self.use_edge_in_head = use_edge_in_head
        head_in = 2 * hidden + (edge_dim if use_edge_in_head else 0)
        self.head = nn.Sequential(nn.Linear(head_in, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, e, mp_src, mp_dst, mp_feat, num_nodes, f_src, f_dst):
        h = Fn.relu(self.embed0(torch.ones(num_nodes, 1, dtype=e.dtype)))
        h = self.l1(h, e, mp_src, mp_dst, mp_feat)
        h = self.l2(h, e, mp_src, mp_dst, mp_feat)
        parts = [h[f_src], h[f_dst]]
        if self.use_edge_in_head:
            parts.append(e)
        return self.head(torch.cat(parts, dim=1)).squeeze(-1)

class EdgeMLP(nn.Module):
    """The control: same features, no graph."""
    def __init__(self, edge_dim, hidden=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(edge_dim, hidden*2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden*2, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1))
    def forward(self, e, *a, **kw):
        return self.net(e).squeeze(-1)

print('E-GraphSAGE params:', sum(p.numel() for p in EGraphSAGE(Xv.shape[1], HIDDEN).parameters()))
print('EdgeMLP     params:', sum(p.numel() for p in EdgeMLP(Xv.shape[1], HIDDEN).parameters()))


## 5 — Batching

A batch is a set of whole windows. Node ids are renumbered inside the batch so
the tensors stay small; because windows share no nodes, this changes nothing
about the computation.


In [ ]:
Xt = torch.from_numpy(Xv)
yt = torch.from_numpy(y)

# flows grouped by window, so a batch is a cheap lookup rather than a scan
order = np.argsort(win_id, kind='stable')
win_starts = np.searchsorted(win_id[order], np.arange(n_win + 1))

def make_batch(window_ids):
    sel = np.concatenate([order[win_starts[w]:win_starts[w+1]] for w in window_ids])
    nodes = np.unique(np.concatenate([flow_src[sel], flow_dst[sel]]))
    remap = np.full(nodes.max() + 1, -1, dtype=np.int64)
    remap[nodes] = np.arange(len(nodes))
    f_s = torch.from_numpy(remap[flow_src[sel]])
    f_d = torch.from_numpy(remap[flow_dst[sel]])
    ar  = torch.arange(len(sel))
    return dict(e=Xt[sel], mp_src=torch.cat([f_s, f_d]), mp_dst=torch.cat([f_d, f_s]),
                mp_feat=torch.cat([ar, ar]), num_nodes=len(nodes),
                f_src=f_s, f_dst=f_d, y=yt[sel], idx=sel)

def batches(window_array, size, shuffle=True, seed=0):
    idx = np.arange(len(window_array))
    if shuffle:
        np.random.default_rng(seed).shuffle(idx)
    for i in range(0, len(idx), size):
        yield window_array[idx[i:i+size]]

b = make_batch(train_wins[:BATCH_WIN])
print(f'sample batch: {len(b["idx"]):,} flows, {b["num_nodes"]:,} nodes, {b["mp_src"].numel():,} messages')


## 6 — Train both models


In [ ]:
n_pos = max(int(y[in_train].sum()), 1)
pos_weight = torch.tensor((int(in_train.sum()) - n_pos) / n_pos, dtype=torch.float32)
print(f'positive class weight: {pos_weight.item():.2f}')

def train(model, name):
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train(); tot = k = 0
        for wsel in batches(train_wins, BATCH_WIN, seed=ep):
            b = make_batch(wsel)
            opt.zero_grad()
            loss = lossf(model(b['e'], b['mp_src'], b['mp_dst'], b['mp_feat'],
                               b['num_nodes'], b['f_src'], b['f_dst']), b['y'])
            loss.backward(); opt.step()
            tot += loss.item(); k += 1
        if (ep + 1) % 5 == 0:
            print(f'  {name} epoch {ep+1:>3}/{EPOCHS}  loss {tot/max(k,1):.4f}'
                  f'  ({(time.time()-t0)/60:.1f} min)')
    print(f'  {name} done in {(time.time()-t0)/60:.1f} min')
    return model

mlp = train(EdgeMLP(Xt.shape[1], HIDDEN, DROPOUT), 'EdgeMLP')
egs = train(EGraphSAGE(Xt.shape[1], HIDDEN, DROPOUT), 'E-GraphSAGE')


## 7 — Results


In [ ]:
@torch.no_grad()
def predict(model, window_array):
    model.eval(); P, T, I = [], [], []
    for wsel in batches(window_array, BATCH_WIN, shuffle=False):
        b = make_batch(wsel)
        P.append(torch.sigmoid(model(b['e'], b['mp_src'], b['mp_dst'], b['mp_feat'],
                                     b['num_nodes'], b['f_src'], b['f_dst'])).numpy())
        T.append(b['y'].numpy()); I.append(b['idx'])
    return np.concatenate(P), np.concatenate(T), np.concatenate(I)

def score(p, t, thr=0.5):
    pr, tr = (p > thr).astype(int), t.astype(int)
    tp = int(((pr==1)&(tr==1)).sum()); fn = int(((pr==0)&(tr==1)).sum())
    fp = int(((pr==1)&(tr==0)).sum()); tn = int(((pr==0)&(tr==0)).sum())
    rec  = tp/max(tp+fn,1); prec = tp/max(tp+fp,1)
    return dict(recall=rec, precision=prec, f1=2*prec*rec/max(prec+rec,1e-9),
                fpr=fp/max(fp+tn,1))

rows = []
store = {}
for name, model in [('EdgeMLP (no graph)', mlp), ('E-GraphSAGE (graph)', egs)]:
    p_s, t_s, _ = predict(model, test_wins)
    p_z, t_z, i_z = predict(model, zero_wins)
    store[name] = (p_z, i_z)
    s = score(p_s, t_s)
    zmask = np.isin(atype[i_z], HELDOUT_ATTACKS)
    zero_recall = float((p_z[zmask] > 0.5).mean())
    rows.append({'Model': name,
                 'Seen recall %':    round(s['recall']*100, 2),
                 'Seen precision %': round(s['precision']*100, 2),
                 'Seen F1 %':        round(s['f1']*100, 2),
                 'Seen FPR %':       round(s['fpr']*100, 2),
                 'Zero-day recall %':round(zero_recall*100, 2)})

res = pd.DataFrame(rows)
display(res)


In [ ]:
gap_f1   = res.loc[1,'Seen F1 %']          - res.loc[0,'Seen F1 %']
gap_zero = res.loc[1,'Zero-day recall %']  - res.loc[0,'Zero-day recall %']
print(f'Graph contribution, seen families : {gap_f1:+.2f} F1 points')
print(f'Graph contribution, zero-day      : {gap_zero:+.2f} recall points')
print()
if gap_f1 > 2 or gap_zero > 2:
    print('The graph adds measurable value over the identical no-graph model.')
else:
    print('The graph adds little at this window size. Try section 8 before')
    print('concluding anything — window size dominates this result.')


### Zero-day recall by family


In [ ]:
rows = []
for fam in HELDOUT_ATTACKS:
    r = {'Family': fam}
    for name, (p_z, i_z) in store.items():
        m = atype[i_z] == fam
        r['Samples'] = int(m.sum())
        r[name] = round(float((p_z[m] > 0.5).mean()*100), 2) if m.sum() else None
    rows.append(r)
per_fam = pd.DataFrame(rows)
display(per_fam)

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120
cols = [c for c in per_fam.columns if c not in ('Family', 'Samples')]
xp = np.arange(len(per_fam)); w = 0.38
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(xp - w/2, per_fam[cols[0]], w, label=cols[0], color='#8fa9c0')
ax.bar(xp + w/2, per_fam[cols[1]], w, label=cols[1], color='#2a9d8f')
ax.set_xticks(xp); ax.set_xticklabels(per_fam['Family'])
ax.set_ylabel('Recall (%)'); ax.set_ylim(0, 100)
ax.set_title('Attack families never seen in training')
ax.legend(); ax.grid(axis='y', alpha=.25)
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/egs_zeroday.png', dpi=150)
plt.show()


## 8 — Window-size sweep *(optional, but the most informative experiment here)*

Re-runs the whole pipeline at several window sizes on a slice of the data.
Expect E-GraphSAGE to improve sharply as the window shrinks, while the MLP
barely moves — the MLP has no graph, so window size cannot affect it.

That contrast is the cleanest evidence that the gain comes from context.


In [ ]:
SWEEP_WINDOWS = [50, 100, 250, 1000]
SWEEP_FLOWS   = 300_000          # a slice, to keep the sweep quick
SWEEP_EPOCHS  = 10

sweep_rows = []
sl = slice(0, min(SWEEP_FLOWS, len(y)))
Xs_, ys_, at_ = Xv[sl], y[sl], atype[sl]
sc_ = pd.factorize(df[SRC_COL].astype(str).values[sl])[0]
dc_ = pd.factorize(df[DST_COL].astype(str).values[sl])[0]

for WIN in SWEEP_WINDOWS:
    fs_, fd_, wi_, nn_ = build_windowed_graph(sc_, dc_, WIN)
    nw_ = int(wi_.max()) + 1
    r2 = np.random.default_rng(SEED)
    w_ = np.arange(nw_); r2.shuffle(w_)
    tr_ = np.sort(w_[:int(0.7*nw_)]); te_ = np.sort(w_[int(0.7*nw_):])
    Xt_, yt_ = torch.from_numpy(Xs_), torch.from_numpy(ys_)
    ordr = np.argsort(wi_, kind='stable')
    ws_ = np.searchsorted(wi_[ordr], np.arange(nw_+1))

    def mb(wids):
        sel = np.concatenate([ordr[ws_[w]:ws_[w+1]] for w in wids])
        nd = np.unique(np.concatenate([fs_[sel], fd_[sel]]))
        rm = np.full(nd.max()+1, -1, dtype=np.int64); rm[nd] = np.arange(len(nd))
        a = torch.from_numpy(rm[fs_[sel]]); b_ = torch.from_numpy(rm[fd_[sel]])
        ar = torch.arange(len(sel))
        return Xt_[sel], torch.cat([a,b_]), torch.cat([b_,a]), torch.cat([ar,ar]), len(nd), a, b_, yt_[sel]

    bw = max(1, 40_000 // WIN)
    pw_ = torch.tensor((ys_[np.isin(wi_, tr_)]==0).sum() /
                        max((ys_[np.isin(wi_, tr_)]==1).sum(), 1), dtype=torch.float32)
    out = {}
    for nm, mk in [('MLP', lambda: EdgeMLP(Xt_.shape[1], HIDDEN, DROPOUT)),
                   ('EGS', lambda: EGraphSAGE(Xt_.shape[1], HIDDEN, DROPOUT))]:
        torch.manual_seed(SEED); m_ = mk()
        o_ = torch.optim.Adam(m_.parameters(), lr=LR, weight_decay=1e-4)
        lf_ = nn.BCEWithLogitsLoss(pos_weight=pw_)
        for ep in range(SWEEP_EPOCHS):
            m_.train()
            for ws2 in batches(tr_, bw, seed=ep):
                e,ms,md,mf,nn2,a,b_,yy = mb(ws2)
                o_.zero_grad(); lf_(m_(e,ms,md,mf,nn2,a,b_), yy).backward(); o_.step()
        m_.eval(); P=[];T=[]
        with torch.no_grad():
            for ws2 in batches(te_, bw, shuffle=False):
                e,ms,md,mf,nn2,a,b_,yy = mb(ws2)
                P.append(torch.sigmoid(m_(e,ms,md,mf,nn2,a,b_)).numpy()); T.append(yy.numpy())
        out[nm] = score(np.concatenate(P), np.concatenate(T))['f1']

    sweep_rows.append({'Window': WIN, 'Windows': nw_,
                       'Flows per host': round(WIN/(nn_/nw_), 1),
                       'MLP F1 %': round(out['MLP']*100, 2),
                       'E-GraphSAGE F1 %': round(out['EGS']*100, 2),
                       'Gain': round((out['EGS']-out['MLP'])*100, 2)})
    print(sweep_rows[-1])

sweep = pd.DataFrame(sweep_rows)
display(sweep)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep['Window'], sweep['MLP F1 %'], 'o--', label='MLP (no graph)', color='#8fa9c0')
ax.plot(sweep['Window'], sweep['E-GraphSAGE F1 %'], 'o-', label='E-GraphSAGE', color='#2a9d8f')
ax.set_xscale('log'); ax.set_xlabel('Window size (flows per graph)')
ax.set_ylabel('F1 (%)'); ax.set_title('Smaller windows give the graph more to say')
ax.legend(); ax.grid(alpha=.25)
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/egs_window_sweep.png', dpi=150)
plt.show()


## 9 — Save


In [ ]:
res.to_csv(f'{OUT_DIR}/egs_comparison.csv', index=False)
per_fam.to_csv(f'{OUT_DIR}/egs_zeroday_by_family.csv', index=False)
if 'sweep' in dir():
    sweep.to_csv(f'{OUT_DIR}/egs_window_sweep.csv', index=False)
torch.save(egs.state_dict(), f'{OUT_DIR}/egraphsage.pt')

summary = {
    'flows'             : int(len(df)),
    'features'          : int(Xt.shape[1]),
    'window'            : WINDOW,
    'windows'           : int(n_win),
    'nodes'             : int(n_nodes),
    'hosts_per_window'  : round(n_nodes/n_win, 1),
    'heldout_families'  : HELDOUT_ATTACKS,
    'graph_gain_f1'     : float(gap_f1),
    'graph_gain_zeroday': float(gap_zero),
}
with open(f'{OUT_DIR}/egs_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print('saved to', os.path.abspath(OUT_DIR))
print(json.dumps(summary, indent=2))


---

## Reading the result

**E-GraphSAGE above the MLP** — context carries information the flow's own
features do not. That is measured evidence for the correlation layer, not an
assertion.

**The two level** — at this window size, neighbouring flows add nothing new.
Run section 8 before drawing that conclusion: window size dominates everything
here, and a single setting is not an answer.

The MLP control is what makes either claim defensible. Without it a higher
E-GraphSAGE number could just be the larger parameter count.

---

### Citation

Lo, W.W., Layeghy, S., Sarhan, M., Gallagher, M., Portmann, M. (2022).
*E-GraphSAGE: A Graph Neural Network based Intrusion Detection System for IoT.*
IEEE/IFIP Network Operations and Management Symposium (NOMS). arXiv:2103.16329

**Declared deviation from the paper:** the flow's own feature vector is appended
to the edge classifier input alongside `CONCAT(z_u, z_v)`. With a testbed of only
a handful of hosts the two endpoint embeddings are near-identical for every flow,
leaving the classifier nothing to separate them with.

**Next (stage 2):** give each edge its timestamp so the model can distinguish
*came before* from *came after* — the step from E-GraphSAGE to a temporal model.
